# Create experiment info

Writes `SAMPLE_DIR/metadata/experiment_info.yaml` — a small, human-readable per-experiment record mirroring the master experiment-info CSVs (e.g. `experiment_info/lt_experiment_info.csv`), so it can later be batch-collected back into one of those tables (`MERci.common.experiment_info.collect_experiment_info`).

Run this after notebook 03 (needs `round_bit_color_map.csv`) and after 01 (needs a bits HAL config for the exposure time).

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd

MERCI_DIR     = Path(os.getcwd()).parent.parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/prepare_imaging/<variant>/<acquisition>/)
SAMPLE_DIR    = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_26/
METADATA_DIR  = SAMPLE_DIR / "metadata"
SETTINGS_DIR  = SAMPLE_DIR / "settings"
POSITIONS_DIR = SAMPLE_DIR / "positions"
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import ExperimentInfo, save_experiment_info
from MERci.acquisition.configs    import read_hal_exposure_time, get_acquisition_type

MICROSCOPE  = "ST2"   # must match what you set in notebooks 01/03
SAMPLE_NAME = SAMPLE_DIR.name

# Acquisition type ("epi" or "disk") is derived from MICROSCOPE, not typed by hand.
ACQUISITION_TYPE = get_acquisition_type(MICROSCOPE)

print(f"SAMPLE_DIR       : {SAMPLE_DIR}")
print(f"SAMPLE_NAME      : {SAMPLE_NAME}")
print(f"ACQUISITION_TYPE : {ACQUISITION_TYPE}")

SAMPLE_DIR       : c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time
SAMPLE_NAME      : 251225_LT027_saving_time
ACQUISITION_TYPE : disk


## Auto-filled fields

Read from what notebooks 01 and 03 already produced, rather than re-typing values that could drift out of sync.

In [ ]:
# Bit count: derived from round_bit_color_map.csv (notebook 03) -- the same
# source notebook 05 uses for N_HYBS.
rbc_path = METADATA_DIR / "round_bit_color_map.csv"
n_bits = None
if rbc_path.exists():
    rbc_df = pd.read_csv(rbc_path)
    n_bits = int(rbc_df["round"].max())
else:
    print(f"WARNING: {rbc_path} not found -- run notebook 03 first. n_bits left as None.")

# Exposure time: read from the bits HAL config XML (notebook 01).
bits_hal_configs = sorted(SETTINGS_DIR.glob("hal-config-*bits*.xml"))
exposure_s = read_hal_exposure_time(bits_hal_configs[-1]) if bits_hal_configs else None
if exposure_s is None:
    print("WARNING: could not read exposure_time from a bits HAL config in settings/.")

# Positions file(s) actually present for this sample (single- or per-tissue layout).
positions_files = sorted(POSITIONS_DIR.glob(f"positions_{SAMPLE_NAME}*.txt"))

print(f"n_bits (hyb rounds) : {n_bits}")
print(f"exposure (ms)       : {exposure_s * 1000 if exposure_s else None}")
print(f"positions files     : {[p.name for p in positions_files]}")

## Fields MERci can't know

Edit these to match the experiment — cluster destination paths and biology/sample metadata that only you know.

In [ ]:
# ── Fields MERci has no way to know -- fill these in ──────────────────────
PROJECT  = "lt"    # "bc" | "lt" | "mf" -- selects which master-CSV schema
                     # this experiment_info.yaml is meant to line up with
LIB_NAME = "LT2"   # codebook/probe-library name, e.g. matches a codebook in
                     # MERci/data/configs/merlin/codebooks/
N_OPT    = 10      # MERlin optimize-iteration count -- independent of bit
                     # count; adjust per experiment/library

# Acquisition-type subfolder between SAMPLE_NAME and this acquisition's own
# MERci/data/metadata/positions/settings (e.g. SAMPLE_DIR/merfish/MERci on the
# microscope computer, .../lineage_tracing/experiments/SAMPLE_DIR/merfish/MERci
# on the cluster) -- "" while this experiment still uses the flat, unsplit
# layout (SAMPLE_DIR itself holds everything). Set to "merfish" once the
# sample folder is split into merfish/lineage siblings.
IMAGING_DIR = ""

# Cluster destination paths (edit the project path / experiment id below --
# these mirror the data_home/merlin_home/folder_name columns in the master
# experiment-info CSVs, e.g. experiment_info/lt_experiment_info.csv)
DATA_HOME   = "/n/holylfs06/LABS/zhuang_lab/Lab/shared/Leonardo/projects/lineage_tracing/experiments"
MERLIN_HOME = (f"{DATA_HOME}/{SAMPLE_NAME}/{IMAGING_DIR}/merlin" if IMAGING_DIR
               else f"{DATA_HOME}/{SAMPLE_NAME}/merlin")
FOLDER_NAME = (f"{SAMPLE_NAME}/{IMAGING_DIR}/data" if IMAGING_DIR
               else f"{SAMPLE_NAME}/data")

# Biology / sample metadata -- anything else the master CSV records that
# MERci has no way to derive from the acquisition itself.
EXTRA = {
    "hyb_temp":         "",   # e.g. "37C, 2d"
    "fix_type":         "",   # e.g. "4%PFA, 15 min"
    "permeabilization": "",   # e.g. "70%EtOH ON"
    "sample_type":      "",   # e.g. embryo stage
    "source":           "",   # institution/collaborator
    "exposure":         exposure_s * 1000 if exposure_s else None,   # ms, auto-filled above
    "n_opt":            N_OPT,   # MERlin optimize-iteration count (see N_OPT above)
    "imaging_dir":      IMAGING_DIR,   # acquisition-type subfolder (see IMAGING_DIR above)
    "positions_name":   [p.name for p in positions_files],
}

print("PROJECT    :", PROJECT)
print("LIB_NAME   :", LIB_NAME)
print("N_OPT      :", N_OPT)
print("IMAGING_DIR:", repr(IMAGING_DIR))
print("DATA_HOME  :", DATA_HOME)
print("MERLIN_HOME:", MERLIN_HOME)
print("FOLDER_NAME:", FOLDER_NAME)
print("EXTRA      :", EXTRA)

## Save

In [4]:
info = ExperimentInfo(
    sample_name      = SAMPLE_NAME,
    project          = PROJECT,
    microscope       = MICROSCOPE,
    acquisition_type = ACQUISITION_TYPE,
    lib_name         = LIB_NAME,
    data_home        = DATA_HOME,
    merlin_home      = MERLIN_HOME,
    folder_name      = FOLDER_NAME,
    extra            = EXTRA,
)

out_path = METADATA_DIR / "experiment_info.yaml"
save_experiment_info(info, out_path)
print(f"Saved: {out_path}\n")
print(out_path.read_text())

Saved: c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\metadata\experiment_info.yaml

sample_name: 251225_LT027_saving_time
project: lt
microscope: ST2
acquisition_type: disk
lib_name: LT2
data_home: /n/holylfs06/LABS/zhuang_lab/Lab/shared/Leonardo/projects/lineage_tracing/experiments
merlin_home: /n/holylfs06/LABS/zhuang_lab/Lab/shared/Leonardo/projects/lineage_tracing/experiments/251225_LT027_saving_time/merlin
folder_name: 251225_LT027_saving_time/merfish/data
hyb_temp: ''
fix_type: ''
permeabilization: ''
sample_type: ''
source: ''
exposure: 250.0
n_opt: 10
positions_name:
- positions_251225_LT027_saving_time.txt
- positions_251225_LT027_saving_time_B1.txt
- positions_251225_LT027_saving_time_B2.txt
- positions_251225_LT027_saving_time_transit_1.txt
- positions_251225_LT027_saving_time_transit_2.txt

